In [ ]:
# Существует отдельный спосо создания модели прямого распространения

In [2]:
import torch
import torchvision
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchvision.datasets import ImageFolder

In [3]:
model = nn.Sequential(
    nn.Linear(28 * 28, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)

# Но в такой записи нету ссылок на отдельные слои модели
print(model)

Sequential(
  (0): Linear(in_features=784, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=10, bias=True)
)


In [4]:
# Их можно создать следующем образом
model = nn.Sequential()
model.add_module('layer_1', nn.Linear(28 * 28, 32))
model.add_module('relu', nn.ReLU())
model.add_module('layer_2', nn.Linear(32, 10))

# Теперь есть переменные слоев
print(model)

Sequential(
  (layer_1): Linear(in_features=784, out_features=32, bias=True)
  (relu): ReLU()
  (layer_2): Linear(in_features=32, out_features=10, bias=True)
)


In [ ]:
# nn.Sequential -> nn.Model, значит nn.Sequential можно использовать как составную часть более сложной модели

In [5]:
# nn.Sequential можно объявить в __init__ модели
class DigitNN(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, 32),
            nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x):
        return self.model(x)

# Или как составную часть другого nn.Sequential
block = nn.Sequential(
    nn.Linear(32, 10),
    nn.ReLU(),
)
model = nn.Sequential()
model.add_module('layer_1', nn.Linear(28 * 28, 32))
model.add_module('block', block )
model.add_module('layer_2', nn.Linear(10, 3))

In [ ]:
# ModuleList необходим для указания слоев при помощи списка

In [ ]:
# Иногда слои в модели необходимо создать через список. В данной реализации:
class ModelNN(nn.Module):
    def __init__(self, input_dim, output_dim, n_layers=3):
        super().__init__()
        self.layers = [nn.Linear(input_dim // n, input_dim // (n+1)) for n in range(1, n_layers+1)]
        sz_input = self.layers[-1].out_features
        self.layer_out = nn.Linear(sz_input, output_dim)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
            x = nn.functional.tanh(x)
        x = self.layer_out(x)
        return x

model = ModelNN(28 * 28, 10)
print(model)

ModelNN(
  (layer_out): Linear(in_features=196, out_features=10, bias=True)
)


In [ ]:
# В модели зарегистрирован только выходно слой. Предыдущие слои остались в списке и не были зарегистрированы
# т.к. на слой не ведет никакая переменная

In [ ]:
# Иногда слои в модели необходимо создать через список. В данной реализации:
class ModelNN(nn.Module):
    def __init__(self, input_dim, output_dim, n_layers=3):
        super().__init__()
        # Можно через nn.Sequential
        # self.layers = [nn.Linear(input_dim // n, input_dim // (n+1)) for n in range(1, n_layers+1)]
        # self.net = nn.Sequential(*self.layers)
        # Но тогда нельзя будет использовать слои по отдельности, а self.net будет отдельной моделью

        self.layers = nn.ModuleList(
            [nn.Linear(input_dim // n, input_dim // (n+1)) for n in range(1, n_layers+1)]
        ) # Для такой записи предусмотрен nn.ModuleList

        sz_input = self.layers[-1].out_features
        self.layer_out = nn.Linear(sz_input, output_dim)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
            x = nn.functional.tanh(x)
        x = self.layer_out(x)
        return x

model = ModelNN(28 * 28, 10)
print(model)

ModelNN(
  (layers): ModuleList(
    (0): Linear(in_features=784, out_features=392, bias=True)
    (1): Linear(in_features=392, out_features=261, bias=True)
    (2): Linear(in_features=261, out_features=196, bias=True)
  )
  (layer_out): Linear(in_features=196, out_features=10, bias=True)
)


In [ ]:
# Удобнее иногда сначала задать nn.ModuleList, а потом добавить в него элементы
class ModelNN(nn.Module):
    def __init__(self, input_dim, output_dim, n_layers=3):
        super().__init__()
        self.layers = nn.ModuleList()
        for n in range(1, n_layers+1):
            # Тогда они будут с подписями
            self.layers.add_module(f'layer_{n}', nn.Linear(input_dim // n, input_dim // (n+1)))

        # sz_input = self.layers[-1].out_features
        # self.layer_out = nn.Linear(sz_input, output_dim)

        self.layer_out = nn.Linear(input_dim // (n_layers+1), output_dim)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
            x = nn.functional.tanh(x)
        x = self.layer_out(x)
        return x

model = ModelNN(28 * 28, 10)
print(model)

ModelNN(
  (layers): ModuleList(
    (0): Linear(in_features=784, out_features=392, bias=True)
    (1): Linear(in_features=392, out_features=261, bias=True)
    (2): Linear(in_features=261, out_features=196, bias=True)
  )
  (layer_out): Linear(in_features=196, out_features=10, bias=True)
)


In [9]:
print(model.layers)

ModuleList(
  (0): Linear(in_features=784, out_features=392, bias=True)
  (1): Linear(in_features=392, out_features=261, bias=True)
  (2): Linear(in_features=261, out_features=196, bias=True)
)


In [10]:
print(model.layers.layer_1)

Linear(in_features=784, out_features=392, bias=True)


In [ ]:
# Класс nn.ModuleDict позволяет хранить элементы сети по ключам

In [ ]:
class ModelNN(nn.Module):
    def __init__(self, input_dim, output_dim, n_layers=3, act_type='relu'):
        super().__init__()
        self.layers = nn.ModuleList()
        self.act_type = act_type
        for n in range(1, n_layers+1):
            self.layers.add_module(f'layer_{n}', nn.Linear(input_dim // n, input_dim // (n+1)))
        self.layer_out = nn.Linear(input_dim // (n_layers+1), output_dim)

        self.act_dict = nn.ModuleDict({
            'relu': nn.ReLU,
            'lk_relu': nn.LeakyReLU,
        })

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
            x = self.act_dict.get(self.act_type, nn.ReLU)(x)
        x = self.layer_out(x)
        return x

model = ModelNN(28 * 28, 10, 'lk_relu')
print(model)